In [1]:
# ============================================================
# [우리단독 모양어댑터 셀 v3.2 - U형+항등, 모드별 예산] 단독 제출 노트북 셀1.
#   v3.2: 시간 예산을 실행 모드로 자동 분기 (품질 설정은 양쪽 다 풀).
#     - 커밋/테스트 (오프라인, eval 120): 항등 35분 + U형 12분 = 약 1시간.
#       빠른 검증용. 싼 태스크부터라 예산 내 최대 커버리지.
#     - 실제 제출 재실행 (리더보드, test 240): 항등 6.7h + U형 3.3h = 약 10h
#       + 로드/파생 오버헤드 -> 12시간 상한 안. TTT 30스텝·16뷰·채점증강 4
#       전부 켜진 정상 품질.
#   (오프라인 풀측정을 다시 하고 싶으면 BUDGET_TEST 를 BUDGET_RERUN 값으로.)
#   v3.1: NLL 채점 배치 분할(4개씩, OOM 시 반감) - 22GB 채점 OOM 10건 해소.
#   v2 -> v3 (정답수 증대):
#     ① 항등(원본) 페이즈 신규: 어댑터 병합 '없이' 순수 sft139 로 전 태스크의
#        원본 문제를 직접 예측 (TTT+16뷰+NLL 동일 적용). 지금까지는 변환판만
#        풀어서 발동 태스크 밖은 후보가 0이었음 - 커버리지 41유닛 -> 전 유닛.
#     ② 2페이즈 서브프로세스: 항등(어댑터 없음) 먼저 -> U형(병합) 나중.
#        프로세스 분리라 GPU 잔류 0 유지.
#     ③ NLL 을 토큰당 평균으로 정규화 - 제출 셀의 kgmon 식(득표-평균점수)
#        에서 득표가 정상 작동 (기존엔 총합 NLL 이 득표를 압도).
#     ④ (제출 셀 v2와 짝) 복원 후 프록시색 잔존 후보 패널티.
#   1등 solver 없이 우리 모델만 돌린다. 짝: 우리단독_제출_셀.py (셀2).
#   Add Input 2개 필요: sorokin/qwen3_4b_grids15_sft139 +
#   kwonyongdeuk/arc2-shapeuni. GPU 노트북.
#   v1 -> v2 (실측 3/41 대응): ① TTT-lite = 쌍둥이마다 chunk 쌍으로
#   소형 LoRA 즉석학습(1등의 태스크별 적응을 이식) ② 뷰 8->16
#   ③ 후보별 증강 NLL 실측(score_aug) -> 선택 손실 감소.
#   (global_end_time 이 1번 셀에서 고정되므로 시간 예산 수정 불필요.
#    병합 셀 v3.5 는 그대로 맨 끝.) GPU 필요.
#   v1.1 -> v1.2: 모델 로드/추론을 서브프로세스로 분리. v1.1 은 셀 안
#   에서 로드해 P/sd/enc/gen 참조가 GPU 0 에 7.1GB 잔류 -> 1등 solver
#   rank0 OOM 전멸 사고 (2026-07-30 로그). 프로세스 종료 = 전부 반환.
#   Add Input 필요: kwonyongdeuk/arc2-shapeuni (pyTorch/shu2val)
#   짝: 2단계 병합 셀 v3.5 (U형 mono 복원 지원. 2c/3c 는 v3.4 S경로 그대로)
#
#   목적: "output 만 collapse" 분해 - input 은 원본 그대로 두고 train output
#   만 모양판으로 바꾼 쌍둥이를 만들어, 통합 모양어댑터(shapeuni) 를 얹은
#   qwen 이 test 모양판을 예측한다. 역색칠(원색 복원)은 병합 셀이 한다.
#   어댑터가 1개라 스왑/라우팅 없음 - 문맥이 변환 종류를 지정한다.
#
#   변형 4종 (코랩 학습 v2 와 동일 정의, 고정 3분류 반영):
#     mono   : output 원색 유지, 고정①색 칸+고정②재등장 도형 -> 단색 m (통일판)
#     mono_a : 위와 같되 고정③ 앵커는 원색 유지 (앵커 보존판, 앵커 있을 때만)
#     3c     : 고정①색 -> m1 / 변동 -> m2
#     2c     : 비배경 전부 -> m (순수 실루엣)
#   등록: mono/mono_a = 종류 'U' (병합 v3.5 신규 복원),
#         2c/3c = 종류 'S' 스펙 (병합 v3.4 부터 있던 S 복원 경로 재사용 -
#         S 복원은 원본 input + 표만 쓰므로 input 원본 쌍둥이에도 그대로 성립)
#   게이트: train(chunk) 전쌍 왕복 복원 일치. 고정/재등장 판정은 모델이
#   문맥으로 보는 chunk 안에서만 도출 (학습과 동일 정보).
#
#   동작 순서:
#     1) U형 쌍둥이 도출 -> 분해_계획.json 등록 (키 {tid}u{j}zzz, 멱등)
#        * challenges_2step.json 에는 넣지 않음 - 1등 기본 실행이 풀 필요 없음
#     2) sft139 로드 + shapeuni LoRA 수동 병합 (peft 불필요)
#     3) 자체 추론: D8+색순열 8뷰 greedy -> 역증강 -> /kaggle/inference_2step
#        에 kgmon 후보 형식(bz2 pickle)으로 덤프 -> 병합 셀이 복원·합류
# ============================================================
MAX_VIRTUAL_U = 9999       # 상한 해제 - 발동하는 전 태스크
PER_TASK_MAX = 3           # 태스크당 쌍둥이 상한 (mono > mono_a > 3c > 2c 순)
N_VIEWS = 16               # 뷰 수 (D8 x 색순열2)
TTT_STEPS = 30             # 쌍둥이별 즉석학습 스텝 (0=끄기)
SCORE_AUGS = 4             # 후보 채점용 증강 문맥 수
ORIG_ENABLE = True         # 항등(원본) 페이즈 on/off
# ---- 시간 예산: (항등, U형) 초. 실행 모드에 따라 자동 선택 ----
BUDGET_RERUN = (24000, 12000)   # 실제 제출 재실행: 총 ~10h (+오버헤드 <12h)
BUDGET_TEST = (2100, 700)       # 커밋/테스트: 총 ~1h (빠른 검증)
import os as _os0
ORIG_BUDGET_S, U_BUDGET_S = (BUDGET_RERUN
                             if _os0.getenv("KAGGLE_IS_COMPETITION_RERUN")
                             else BUDGET_TEST)
print(f"예산 모드: {'제출 재실행(풀)' if _os0.getenv('KAGGLE_IS_COMPETITION_RERUN') else '커밋 테스트(1h)'} "
      f"- 항등 {ORIG_BUDGET_S}s / U형 {U_BUDGET_S}s")
LIMIT = 0                  # 0=전체(120태스크 발동분) / 1=가장 쉬운 쌍둥이 1개만
CTX_CELLS = 2200           # 문맥 train 쌍 셀 예산 (학습 chunk 2800 보다 보수적)
GEN_CELL_MAX = 1100        # 생성 토큰 상한

import os, re, json, glob, time, bz2, pickle, zlib
import numpy as np

# ---------------- 공통 유틸 (병합 셀과 동일 정의) ----------------
def _G(x):
    return np.asarray(x, dtype=int)

def _shapes(g, bg, conn=4):
    g = _G(g); H, W = g.shape
    NB = ((1, 0), (-1, 0), (0, 1), (0, -1)) if conn == 4 else \
         tuple((a, b) for a in (-1, 0, 1) for b in (-1, 0, 1) if (a, b) != (0, 0))
    seen = np.zeros((H, W), bool); out = []
    for y in range(H):
        for x in range(W):
            if seen[y, x] or g[y, x] == bg:
                continue
            c0 = int(g[y, x]); st = [(y, x)]; seen[y, x] = True; cs = []
            while st:
                a, b = st.pop(); cs.append((a, b))
                for dy, dx in NB:
                    ny, nx = a + dy, b + dx
                    if 0 <= ny < H and 0 <= nx < W and not seen[ny, nx] \
                            and g[ny, nx] == c0:
                        seen[ny, nx] = True; st.append((ny, nx))
            out.append((c0, cs))
    return out

def _norm(cs):
    y0 = min(y for y, _ in cs); x0 = min(x for _, x in cs)
    return frozenset((y - y0, x - x0) for y, x in cs)

def _center(cs):
    return (sum(y for y, _ in cs) / len(cs), sum(x for _, x in cs) / len(cs))

def _S_silkey(cs):
    return json.dumps(sorted(map(list, _norm(cs))))

# ---------------- 변환 정의 (코랩 학습 v2 와 동일 - 0 = 배경) ----------------
class DiagonalShapeExtractor:
    def extract_all_shapes(self, grid):
        if not grid or len(grid) == 0:
            return []
        arr = np.array(grid); shapes = []
        for color in np.unique(arr):
            if color == 0:
                continue
            coords = np.argwhere(arr == color)
            if len(coords) == 0:
                continue
            shapes.append({'pixels': [(int(y), int(x)) for y, x in coords],
                           'color': int(color)})
        return shapes

    def extract_all_regions(self, grid):
        if not grid or len(grid) == 0:
            return []
        from scipy import ndimage
        arr = np.array(grid); regions = []
        for color in np.unique(arr):
            if color == 0:
                continue
            labeled, n = ndimage.label(arr == color)
            for rid in range(1, n + 1):
                px = [(int(y), int(x)) for y, x in np.argwhere(labeled == rid)]
                if px:
                    regions.append({'pixels': px, 'color': int(color)})
        return regions

_DSE = DiagonalShapeExtractor()

def _unit_key(u):
    px = u['pixels']
    y0 = min(y for y, _ in px); x0 = min(x for _, x in px)
    return (u['color'], frozenset((y - y0, x - x0) for y, x in px))

def _cols_of(chunk):
    used = set()
    for p in chunk:
        used |= set(np.unique(_G(p['input'])).tolist())
        used |= set(np.unique(_G(p['output'])).tolist())
    return {int(c) for c in used}

def _free(chunk, n):
    used = _cols_of(chunk)
    fr = [c for c in range(9, 0, -1) if c not in used]
    return fr[:n] if len(fr) >= n else None

def _fixed_cols(chunk):
    F = []
    for c in sorted(_cols_of(chunk) - {0}):
        ok, present = True, False
        for p in chunk:
            i, o = _G(p['input']), _G(p['output'])
            if i.shape != o.shape or not np.array_equal(i == c, o == c):
                ok = False; break
            if (i == c).any():
                present = True
        if ok and present:
            F.append(c)
    return F

def _partial_anchor_cells(i, o):
    if i.shape != o.shape:
        return set()
    anchors = set()
    try:
        out_pos = {(u['color'], frozenset(u['pixels']))
                   for u in _DSE.extract_all_regions(o.tolist())}
        for u in _DSE.extract_all_regions(i.tolist()):
            c = u['color']; key = (c, frozenset(u['pixels']))
            if key not in out_pos:
                continue
            if np.array_equal(i == c, o == c):
                continue
            anchors.add(key)
    except Exception:
        pass
    return anchors

def _mono_mask(p, F, anchor_keep):
    i, o = _G(p['input']), _G(p['output'])
    mono = np.zeros(o.shape, bool)
    if F and i.shape == o.shape:
        mono |= np.isin(o, F)
    try:
        in_keys = {_unit_key(u) for u in _DSE.extract_all_regions(i.tolist())} | \
                  {_unit_key(u) for u in _DSE.extract_all_shapes(i.tolist())}
        o_list = o.tolist()
        for u in (_DSE.extract_all_regions(o_list)
                  + _DSE.extract_all_shapes(o_list)):
            if _unit_key(u) in in_keys:
                for y, x in u['pixels']:
                    mono[y, x] = True
    except Exception:
        pass
    if anchor_keep:
        for c, px in _partial_anchor_cells(i, o):
            for y, x in px:
                mono[y, x] = False
    return mono

def _tf_mono(chunk, keep_anchor):
    fr = _free(chunk, 1)
    if not fr:
        return None
    m = fr[0]; F = _fixed_cols(chunk)
    out, changed, has_anchor = [], False, False
    for p in chunk:
        i, o = _G(p['input']), _G(p['output'])
        if keep_anchor and _partial_anchor_cells(i, o):
            has_anchor = True
        mono = _mono_mask(p, F, anchor_keep=keep_anchor)
        if mono.any():
            changed = True
        o2 = o.copy(); o2[mono] = m
        out.append({'input': p['input'], 'output': o2.tolist()})
    if keep_anchor and not has_anchor:
        return None
    return (out, m, F) if changed else None

def _tf_2c(chunk):
    if len(_cols_of(chunk) - {0}) < 2:
        return None
    fr = _free(chunk, 1)
    if not fr:
        return None
    m = fr[0]
    out = [{'input': p['input'],
            'output': np.where(_G(p['output']) != 0, m, 0).tolist()}
           for p in chunk]
    return out, m

def _tf_3c(chunk):
    F = _fixed_cols(chunk)
    rest = sorted(_cols_of(chunk) - {0} - set(F))
    if not F or not rest:
        return None
    fr = _free(chunk, 2)
    if not fr:
        return None
    m1, m2 = fr[0], fr[1]
    out = []
    for p in chunk:
        o = _G(p['output']).copy()
        fmask = np.isin(o, F); vmask = (o != 0) & ~fmask
        o[fmask] = m1; o[vmask] = m2
        out.append({'input': p['input'], 'output': o.tolist()})
    return out, m1, m2, F, rest

# ---------------- 복원 (게이트/오프라인 측정용 - 병합 셀과 동일 로직) --------
def _S_sil_paint(p, i, bg, m, colors):
    if not colors:
        return
    pool = {}
    for c, cs in _shapes(i, bg):
        if c in colors:
            pool.setdefault(_norm(cs), []).append((c, cs))
    for cs in [cs for c, cs in _shapes(p, bg) if c == m]:
        cand = pool.get(_norm(cs), [])
        if not cand:
            continue
        if len({c for c, _ in cand}) == 1:
            used = cand[0]
        else:
            cy, cx = _center(cs)
            used = min(cand, key=lambda t: (_center(t[1])[0] - cy) ** 2
                                           + (_center(t[1])[1] - cx) ** 2)
        cand.remove(used)
        for y, x in cs:
            p[y, x] = used[0]

def _S_groups(spec):
    v = spec['변형']; F, M, V = spec['F'], spec['M'], spec['V']
    if v == '2색':
        return [], F + M + V
    if v == '3색':
        return F, M + V
    if v == '3색도형':
        return F + M, V
    return [], M + V

def _restore_S(pred, spec, orig_in):
    p = _G(pred).copy(); i = _G(orig_in); bg = spec['bg']
    G1, G2 = _S_groups(spec)
    m1, m2 = spec.get('m1'), spec['m2']
    same = (p.shape == i.shape)
    if m1 is not None and same:
        mask = (p == m1) & np.isin(i, spec['F'])
        p[mask] = i[mask]
    if m1 is not None:
        _S_sil_paint(p, i, bg, m1, [c for c in spec['M'] if c in G1])
    mode = spec['변동복원']
    if mode != '모양색':
        _S_sil_paint(p, i, bg, m2, G2)
    if mode == '모양색':
        f = spec['모양표']
        for cs in [cs for c, cs in _shapes(p, bg) if c == m2]:
            v = f.get(_S_silkey(cs))
            if v is not None:
                for y, x in cs:
                    p[y, x] = v
    if mode in ('위치', '위치단색') and same:
        mask = (p == m2) & np.isin(i, G2)
        p[mask] = i[mask]
    if mode == '전이' and same:
        f = spec['전이표']
        ys, xs = np.where(p == m2)
        for y, x in zip(ys.tolist(), xs.tolist()):
            v = f.get(str(int(i[y, x])))
            if v is not None:
                p[y, x] = v
    if mode in ('단색', '위치단색') and spec.get('단색') is not None:
        p[p == m2] = spec['단색']
    return p

def _restore_U(pred, spec, orig_in):
    """U형(mono 계열): m 칸 = ①input F색 자리 되칠 ②잔여 m 성분은 input
    도형(4연결, bg=0) 실루엣 매칭 (동형 다색 -> 위치 최근접)"""
    p = _G(pred).copy(); i = _G(orig_in); m = spec['m']
    if p.shape == i.shape and spec['F']:
        mask = (p == m) & np.isin(i, spec['F'])
        p[mask] = i[mask]
    _S_sil_paint(p, i, 0, m, sorted({int(c) for c in np.unique(i) if c != 0}))
    return p

def restore_any(pred, spec, orig_in):
    if spec['종류'] == 'I':
        return _G(pred)
    if spec['종류'] == 'U':
        return _restore_U(pred, spec, orig_in)
    return _restore_S(pred, spec, orig_in)

# ---------------- 스펙 도출 (chunk 기준) + 왕복 게이트 ----------------
def _var_restore_tables(chunk, spec):
    """m2 변동복원 모드 도출 (S셀과 동일: 단색/전이/모양색/위치/위치단색).
    통과 스펙 반환 또는 None"""
    same_all = all(_G(p['input']).shape == _G(p['output']).shape
                   for p in chunk)
    def apply_out(o):
        o = _G(o).copy()
        G1, G2 = _S_groups(spec)
        if G1:
            o[np.isin(o, G1)] = spec['m1']
        if G2:
            o[np.isin(o, G2)] = spec['m2']
        return o
    solid, trans, silmap = set(), ({} if same_all else None), {}
    for p in chunk:
        i, o = _G(p['input']), _G(p['output'])
        to = apply_out(o)
        m2m = (to == spec['m2'])
        solid |= {int(v) for v in o[m2m]}
        if trans is not None:
            for a, b in zip(i[m2m].tolist(), o[m2m].tolist()):
                k = str(int(a))
                if trans.get(k, int(b)) != int(b):
                    trans = None; break
                trans[k] = int(b)
        if silmap is not None:
            for c, cs in _shapes(to, spec['bg']):
                if c != spec['m2']:
                    continue
                oc = {int(o[y, x]) for y, x in cs}
                if len(oc) != 1:
                    silmap = None; break
                k = _S_silkey(cs); v = oc.pop()
                if silmap.get(k, v) != v:
                    silmap = None; break
                silmap[k] = v
    c_star = int(next(iter(solid))) if len(solid) == 1 else None
    modes = (['단색'] if c_star is not None else []) + \
            (['전이'] if trans else []) + \
            (['모양색'] if silmap else []) + ['위치'] + \
            (['위치단색'] if c_star is not None else [])
    for mode in modes:
        sp = dict(spec); sp['변동복원'] = mode; sp['단색'] = c_star
        if mode == '전이':
            sp['전이표'] = trans
        if mode == '모양색':
            sp['모양표'] = silmap
        ok = True
        for p in chunk:
            try:
                to = apply_out(p['output'])
                if not np.array_equal(_restore_S(to, sp, p['input']),
                                      _G(p['output'])):
                    ok = False; break
            except Exception:
                ok = False; break
        if ok:
            return sp
    return None

def derive_U_twins(chunk):
    """[(spec, 변환된 chunk, 꼬리표)] - 우선순위 mono > mono_a > 3c > 2c"""
    out = []
    r = _tf_mono(chunk, keep_anchor=False)                 # mono (통일판)
    if r:
        tc, m, F = r
        sp = {'종류': 'U', '변형': 'mono', 'F': F, 'm': m, 'bg': 0}
        if all(np.array_equal(_restore_U(t['output'], sp, p['input']),
                              _G(p['output'])) for t, p in zip(tc, chunk)):
            out.append((sp, tc, 'Umono'))
    r = _tf_mono(chunk, keep_anchor=True)                  # mono_a (앵커 보존)
    if r:
        tc, m, F = r
        sp = {'종류': 'U', '변형': 'mono_a', 'F': F, 'm': m, 'bg': 0}
        if all(np.array_equal(_restore_U(t['output'], sp, p['input']),
                              _G(p['output'])) for t, p in zip(tc, chunk)):
            out.append((sp, tc, 'Umono_a'))
    r = _tf_3c(chunk)                                      # 3c -> S'3색' 스펙
    if r:
        tc, m1, m2, F, V = r
        sp0 = {'종류': 'S', '변형': '3색', 'F': F, 'M': [], 'V': V,
               'bg': 0, 'm1': m1, 'm2': m2}
        sp = _var_restore_tables(chunk, sp0)
        if sp:
            out.append((sp, tc, f"U3c/{sp['변동복원']}"))
    r = _tf_2c(chunk)                                      # 2c -> S'2색' 스펙
    if r:
        tc, m = r
        cols = sorted(_cols_of(chunk) - {0})
        sp0 = {'종류': 'S', '변형': '2색', 'F': [], 'M': [], 'V': cols,
               'bg': 0, 'm1': None, 'm2': m}
        sp = _var_restore_tables(chunk, sp0)
        if sp:
            out.append((sp, tc, f"U2c/{sp['변동복원']}"))
    # 동일 변환판 중복 제거 (mono == mono_a 가 되는 경우 등)
    seen, ded = set(), []
    for sp, tc, tag in out:
        key = json.dumps([t['output'] for t in tc], sort_keys=True)
        if key in seen:
            continue
        seen.add(key); ded.append((sp, tc, tag))
    return ded

def pick_chunk(train):
    """문맥 = 학습 chunk 와 동일 조건 (<=4쌍, 셀 예산). 결정적 선택."""
    sel, cells = [], 0
    for p in train:
        if len(sel) >= 4:
            break
        c = (len(p['input']) * len(p['input'][0]) +
             len(p['output']) * len(p['output'][0]))
        if c > CTX_CELLS or cells + c > CTX_CELLS:
            continue
        sel.append(p); cells += c
    return sel if len(sel) >= 2 else None

# ---------------- 1) 쌍둥이 도출 + 분해_계획 등록 (멱등) ----------------
_rerun = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
_BASE = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/"
_SRC = _BASE + ("arc-agi_test_challenges.json" if _rerun
                else "arc-agi_evaluation_challenges.json")
_CH = json.load(open(_SRC))
try:
    _plan = json.load(open("/kaggle/working/분해_계획.json"))
except Exception:
    _plan = {}

for _tid in list(_plan):                      # 기존 U 쌍둥이 제거 (멱등)
    _plan[_tid] = [tw for tw in _plan[_tid]
                   if not re.match(r".*[uo]\d+zzz$", tw["k"])]
    if not _plan[_tid]:
        _plan.pop(_tid)

TWINS = []                                    # (key, tid, spec, chunk_t, tag)
for _tid in sorted(_CH):
    if sum(1 for t in TWINS if t[1] == _tid) >= PER_TASK_MAX:
        continue
    train = _CH[_tid].get('train', [])
    chunk = pick_chunk(train)
    if chunk is None:
        continue
    try:
        cands = derive_U_twins(chunk)
    except Exception:
        continue
    _j = 0
    for sp, tc, tag in cands[:PER_TASK_MAX]:
        if len(TWINS) >= MAX_VIRTUAL_U:
            break
        key = f"{_tid}u{_j}zzz"; _j += 1
        TWINS.append((key, _tid, sp, tc, tag))
        _plan.setdefault(_tid, []).append({"k": key, "spec": sp, "tag": tag})
    if len(TWINS) >= MAX_VIRTUAL_U:
        break

OTW = []                                      # 항등(원본) - 전 태스크
if ORIG_ENABLE:
    for _tid in sorted(_CH):
        train = _CH[_tid].get('train', [])
        chunk = pick_chunk(train)
        if chunk is None:                     # 폴백: 작은 쌍부터 최대 3000셀
            prs = sorted(train, key=lambda p:
                         len(p['input']) * len(p['input'][0]) +
                         len(p['output']) * len(p['output'][0]))
            sel, cells = [], 0
            for p in prs:
                c = (len(p['input']) * len(p['input'][0]) +
                     len(p['output']) * len(p['output'][0]))
                if cells + c > 3000 or len(sel) >= 4:
                    break
                sel.append(p); cells += c
            chunk = sel or None
        if chunk is None:
            continue
        key = f"{_tid}o0zzz"
        sp = {'종류': 'I'}
        OTW.append((key, _tid, sp, chunk, 'orig'))
        _plan.setdefault(_tid, []).append({"k": key, "spec": sp, "tag": 'orig'})

with open("/kaggle/working/분해_계획.json", "w") as _f:
    json.dump(_plan, _f)

def _twin_cost(t):
    """쉬운(작은) 것부터: 문맥 셀 수 + test 셀 수*2 오름차순"""
    _, tid, _, tc, _ = t
    c = sum(len(x['input']) * len(x['input'][0]) +
            len(x['output']) * len(x['output'][0]) for x in tc)
    c += max(len(te['input']) * len(te['input'][0])
             for te in _CH[tid]['test']) * 2
    return c
TWINS.sort(key=_twin_cost)                    # 예산 안에서 최대한 많이
OTW.sort(key=_twin_cost)
from collections import Counter
print(f"U형 쌍둥이 {len(TWINS)}개 ({len({t[1] for t in TWINS})}태스크): "
      f"{dict(Counter(t[4].split('/')[0] for t in TWINS))} "
      f"/ 항등 {len(OTW)}태스크")

# ---------------- 2) 추론 서브프로세스 (GPU 잔류 0 보장) ----------------
_INFER_SRC = r'''# shape_infer.py v2 - TTT-lite + 16뷰 + 후보 NLL 채점
import os, json, glob, time, bz2, pickle, zlib
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

PL = json.load(open('/kaggle/working/shape_payload.json'))
N_VIEWS = PL['views']; BUDGET_S = PL['budget']
TTT_STEPS = PL.get('ttt_steps', 0); SCORE_AUGS = PL.get('score_augs', 4)

_cfgs = sorted({os.path.dirname(p) for p in glob.glob(
    '/kaggle/input/**/config.json', recursive=True)})
_cfgs.sort(key=lambda d: 'sft139' not in d)
MODEL_DIR = _cfgs[0] if _cfgs else None
assert MODEL_DIR and 'sft139' in MODEL_DIR, f'sft139 없음: {_cfgs[:3]}'
tok = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
tok.padding_side = 'left'
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
mdl = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, torch_dtype=torch.bfloat16, local_files_only=True).cuda().eval()

# tied embed/lm_head 분리 (LoRA 델타 겹침 방지)
if PL.get('adapter') and getattr(mdl.config, 'tie_word_embeddings', False):
    import torch.nn as nn
    W0 = mdl.get_input_embeddings().weight
    nh = nn.Linear(W0.shape[1], W0.shape[0], bias=False,
                   dtype=W0.dtype, device=W0.device)
    nh.weight.data = W0.data.clone()
    mdl.set_output_embeddings(nh)
    mdl.config.tie_word_embeddings = False
    print('tied lm_head 분리 완료')

# shapeuni 전역 어댑터를 가중치에 병합 (베이스로 흡수)
if PL.get('adapter'):
    sd = torch.load(PL['adapter'], map_location='cpu')
    SCALE = 32.0 / 256.0
    P = dict(mdl.named_parameters())
    n_mg = 0
    with torch.no_grad():
        for k in list(sd):
            if 'lora_A' not in k and 'lora_embedding_A' not in k:
                continue
            k2 = k.replace('.default', '')
            emb = 'lora_embedding_A' in k2
            kb = k2.replace('lora_embedding_A', 'lora_embedding_B') if emb \
                 else k2.replace('lora_A', 'lora_B')
            base = k2.split('.lora_')[0].replace('base_model.model.', '')
            W = P.get(base + '.weight')
            assert W is not None, '병합 대상 없음: ' + base
            d = (sd[kb].float() @ sd[k].float())
            if emb:
                d = d.T
            assert d.shape == W.shape, f'{base}: {d.shape} vs {W.shape}'
            W += (SCALE * d).to(W.dtype).to(W.device)
            n_mg += 1
    print('LoRA 병합 %d층 <- %s' % (n_mg, os.path.basename(PL['adapter'])))
else:
    print('항등 페이즈: 어댑터 병합 없음 (순수 sft139)')
IM_END = tok.convert_tokens_to_ids('<|im_end|>')
ASST = tok.encode('<|im_start|>assistant\n', add_special_tokens=False)
ENDT = tok.encode('<|im_end|>', add_special_tokens=False)[0]

# TTT-lite 준비: 소형 LoRA (쌍둥이마다 리셋)
HAVE_TTT = TTT_STEPS > 0
if HAVE_TTT:
    try:
        from peft import (LoraConfig, get_peft_model,
                          get_peft_model_state_dict, set_peft_model_state_dict)
        try:
            mdl.enable_input_require_grads()
        except Exception:
            pass
        mdl.gradient_checkpointing_enable()
        lcfg = LoraConfig(r=32, lora_alpha=32, lora_dropout=0.0, bias='none',
                          target_modules=['q_proj', 'k_proj', 'v_proj',
                                          'o_proj', 'gate_proj', 'up_proj',
                                          'down_proj'])
        mdl = get_peft_model(mdl, lcfg)
        INIT = {k: v.clone() for k, v in get_peft_model_state_dict(mdl).items()}
        print('TTT-lite 준비: LoRA r=32, %d스텝/쌍둥이' % TTT_STEPS)
    except Exception as e:
        print('peft 사용 불가 -> TTT 생략:', str(e)[:120])
        HAVE_TTT = False

def _gs(g):
    return '\n'.join(''.join(str(int(v)) for v in row) for row in g)

def _aug(g, k, fl, perm):
    a = np.rot90(np.asarray(g, dtype=int), k)
    if fl:
        a = np.fliplr(a)
    return perm[a]

def _deaug(g, k, fl, perm):
    ip = np.zeros(10, int); ip[perm] = np.arange(10)
    a = ip[np.asarray(g, dtype=int)]
    if fl:
        a = np.fliplr(a)
    return np.rot90(a, -k)

def _parse(txt):
    txt = txt.split('<|im_end|>')[0].strip()
    rows = [r.strip() for r in txt.split('\n') if r.strip()]
    if not rows or any(not r.isdigit() for r in rows):
        return None
    w = len(rows[0])
    if any(len(r) != w for r in rows) or len(rows) > 30 or w > 30:
        return None
    return [[int(ch) for ch in r] for r in rows]

def _ctx_text(chunk, k, fl, perm, order=None):
    t = ''
    for oi in (order if order is not None else range(len(chunk))):
        x = chunk[oi]
        t += ('<|im_start|>user\n' + _gs(_aug(x['input'], k, fl, perm))
              + '<|im_end|><|im_start|>assistant\n'
              + _gs(_aug(x['output'], k, fl, perm)) + '<|im_end|>')
    return t

def _labels(ids):
    lab = [-100] * len(ids); j = 0
    while j <= len(ids) - len(ASST):
        if ids[j:j + len(ASST)] == ASST:
            s = j + len(ASST); e = s
            while e < len(ids) and ids[e] != ENDT:
                e += 1
            for t in range(s, min(e + 1, len(ids))):
                lab[t] = ids[t]
            j = e
        j += 1
    return lab

def _pad(seqs, labs=None):
    ml = max(len(x) for x in seqs); pad = tok.pad_token_id or 0
    ii = torch.full((len(seqs), ml), pad, dtype=torch.long)
    am = torch.zeros((len(seqs), ml), dtype=torch.long)
    lb = torch.full((len(seqs), ml), -100, dtype=torch.long)
    for i, s in enumerate(seqs):
        ii[i, :len(s)] = torch.tensor(s); am[i, :len(s)] = 1
        if labs is not None:
            lb[i, :len(labs[i])] = torch.tensor(labs[i])
    return ii.cuda(), am.cuda(), lb.cuda()

def _ttt(chunk, rng):
    """쌍둥이별 즉석학습: chunk 쌍을 D8+색순열+순서셔플 증강해 소형 LoRA 학습"""
    set_peft_model_state_dict(mdl, {k: v.clone() for k, v in INIT.items()})
    texts = []
    for _ in range(24):
        k, fl = int(rng.randint(4)), bool(rng.randint(2))
        perm = np.concatenate([[0], rng.permutation(np.arange(1, 10))])
        texts.append(_ctx_text(chunk, k, fl, perm,
                               order=rng.permutation(len(chunk))))
    seqs = [tok.encode(t)[:3600] for t in texts]
    labs = [_labels(s) for s in seqs]
    mdl.train()
    opt = torch.optim.AdamW([p for p in mdl.parameters() if p.requires_grad],
                            lr=1e-4)
    bi = 0
    for _ in range(TTT_STEPS):
        idx = [(bi + z) % len(seqs) for z in range(2)]; bi += 2
        ii, am, lb = _pad([seqs[z] for z in idx], [labs[z] for z in idx])
        out = mdl(input_ids=ii, attention_mask=am, labels=lb)
        out.loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
    mdl.eval()
    del opt
    torch.cuda.empty_cache()

@torch.no_grad()
def _nll(chunk, test_in, grids, rng):
    """각 후보의 증강 문맥 '토큰당' NLL (v3.1: 배치 분할 + OOM 적응.
    구판은 후보 전부 한 배치 -> 22GB GPU 에서 채점 OOM 10건 발생)"""
    res = [[] for _ in grids]
    for a in range(SCORE_AUGS):
        k, fl = a % 4, a >= 4
        perm = np.concatenate([[0], rng.permutation(np.arange(1, 10))])
        q = (_ctx_text(chunk, k, fl, perm) + '<|im_start|>user\n'
             + _gs(_aug(test_in, k, fl, perm))
             + '<|im_end|><|im_start|>assistant\n')
        qt = tok.encode(q)
        seqs, spans = [], []
        for g in grids:
            at = tok.encode(_gs(_aug(g, k, fl, perm)) + '<|im_end|>')
            seqs.append(qt + at); spans.append((len(qt), len(at)))
        gi, bs = 0, 4
        while gi < len(seqs):
            j2 = min(gi + bs, len(seqs))
            try:
                ii, am, _ = _pad(seqs[gi:j2])
                logits = mdl(input_ids=ii, attention_mask=am).logits.float()
                logp = torch.log_softmax(logits, -1).cpu()
                del logits
                for z in range(j2 - gi):
                    s, n = spans[gi + z]
                    tgt = torch.tensor(seqs[gi + z][s:s + n])
                    lp = logp[z, s - 1:s - 1 + n]
                    res[gi + z].append(
                        float(-lp[torch.arange(n), tgt].sum()) / max(n, 1))
                gi = j2
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs > 1:
                    bs = max(1, bs // 2)       # 배치 반감 후 재시도
                else:
                    res[gi].append(1.0)        # 1개도 안 되면 중립값
                    gi += 1
        torch.cuda.empty_cache()
    return res

OUTD = '/kaggle/inference_2step'
os.makedirs(OUTD, exist_ok=True)
t0 = time.time(); n_dump = n_unit = 0
for tw in PL['twins']:
    if time.time() - t0 > BUDGET_S:
        print('시간 예산 소진 -', tw['key'], '부터 생략'); break
    rng = np.random.RandomState(zlib.crc32(tw['key'].encode()) % (2 ** 31))
    if HAVE_TTT:
        try:
            ts = time.time(); _ttt(tw['chunk'], rng)
            print('  TTT %s %ds' % (tw['key'], int(time.time() - ts)))
        except Exception as e:
            print('  TTT 실패(생성은 진행) %s: %s' % (tw['key'], str(e)[:80]))
            torch.cuda.empty_cache()
    for ti_idx, test_in in enumerate(tw['tests']):
        prompts, views = [], []
        for v in range(N_VIEWS):
            k, fl = v % 4, (v // 4) % 2 == 1
            perm = np.concatenate([[0], rng.permutation(np.arange(1, 10))])
            t = (_ctx_text(tw['chunk'], k, fl, perm)
                 + '<|im_start|>user\n' + _gs(_aug(test_in, k, fl, perm))
                 + '<|im_end|><|im_start|>assistant\n')
            prompts.append(t); views.append((k, fl, perm))
        sols = []                                  # 뷰별 원방향 모양판
        try:
            for b0 in range(0, N_VIEWS, 8):        # 8개씩 나눠 생성
                enc = tok(prompts[b0:b0 + 8], return_tensors='pt',
                          padding=True)
                enc = {a: b.cuda() for a, b in enc.items()}
                with torch.no_grad():
                    gen = mdl.generate(**enc, do_sample=False,
                                       max_new_tokens=tw['cap'],
                                       eos_token_id=IM_END,
                                       pad_token_id=tok.pad_token_id,
                                       use_cache=True)
                new = gen[:, enc['input_ids'].shape[1]:]
                for vi, row in enumerate(new):
                    g = _parse(tok.decode(row, skip_special_tokens=False))
                    if g is None:
                        continue
                    k, fl, perm = views[b0 + vi]
                    try:
                        sols.append(_deaug(g, k, fl, perm).tolist())
                    except Exception:
                        continue
        except Exception as e:
            print('  %s_%d 추론 실패: %s' % (tw['key'], ti_idx, str(e)[:80]))
            torch.cuda.empty_cache(); continue
        if not sols:
            continue
        uniq = []
        for g in sols:                             # 고유 후보 -> NLL 채점
            if g not in uniq:
                uniq.append(g)
        try:
            sc = _nll(tw['chunk'], test_in, uniq, rng)
        except Exception as e:
            print('  %s_%d 채점 실패(1.0 대체): %s'
                  % (tw['key'], ti_idx, str(e)[:80]))
            torch.cuda.empty_cache()
            sc = [[1.0]] * len(uniq)
        outs = [{'solution': g, 'score_aug': sc[uniq.index(g)]} for g in sols]
        with bz2.BZ2File('%s/%s_%d.shape' % (OUTD, tw['key'], ti_idx),
                         'w') as f:
            pickle.dump(outs, f)
        n_dump += len(outs); n_unit += 1
print('추론 완료: %d유닛 / 후보 %d개 덤프 (%ds, TTT=%s)'
      % (n_unit, n_dump, int(time.time() - t0), HAVE_TTT))
'''

if LIMIT:
    TWINS = TWINS[:LIMIT]
    OTW = OTW[:LIMIT]
ADPT = sorted(glob.glob("/kaggle/input/**/shapeuni_*.pt", recursive=True))
if not ADPT:
    print("*** shapeuni_*.pt 없음: Add Input -> Models -> "
          "kwonyongdeuk/arc2-shapeuni 추가하라. U형 페이즈 생략 ***")

with open('/kaggle/working/shape_infer.py', 'w') as _f:
    _f.write(_INFER_SRC)
import subprocess, sys

def _run_phase(twins, budget, adapter, label):
    """페이즈별 서브프로세스 (프로세스 종료 = GPU 전부 반환)"""
    if not twins or budget <= 0:
        print(f'[{label}] 대상 없음/예산 0 - 생략'); return
    payload = {'budget': budget, 'views': N_VIEWS, 'adapter': adapter,
               'ttt_steps': TTT_STEPS, 'score_augs': SCORE_AUGS, 'twins': []}
    for key, tid, sp, tc, tag in twins:
        mx = max(len(t['output']) * (len(t['output'][0]) + 1) for t in tc)
        payload['twins'].append(
            {'key': key, 'tag': tag,
             'cap': min(GEN_CELL_MAX, int(mx * 1.5) + 32),
             'chunk': tc,
             'tests': [te['input'] for te in _CH[tid]['test']]})
    with open('/kaggle/working/shape_payload.json', 'w') as _f:
        json.dump(payload, _f)
    _env = dict(os.environ, TOKENIZERS_PARALLELISM='false')
    _t = time.time()
    _rc = subprocess.run([sys.executable, 'shape_infer.py'],
                         cwd='/kaggle/working', env=_env).returncode
    print(f'[{label}] 페이즈 종료 rc={_rc} ({int(time.time() - _t)}s)')

_run_phase(OTW, ORIG_BUDGET_S, None, '항등')     # ① 순수 sft139 - 전 태스크
if ADPT:
    _run_phase(TWINS, U_BUDGET_S, ADPT[0], 'U형')  # ② shapeuni 병합
RUN = bool(TWINS or OTW)

# ---------------- 3) 오프라인 GT 대조 (CPU, 메인 셀) ----------------
if RUN and not _rerun:
    try:
        _SOL = json.load(open(_BASE + 'arc-agi_evaluation_solutions.json'))
    except Exception:
        _SOL = None
    if _SOL:
        n_hit = n_unit = 0
        for key, tid, sp, tc, tag in (OTW + TWINS):
            for ti_idx, te in enumerate(_CH[tid]['test']):
                fp = f'/kaggle/inference_2step/{key}_{ti_idx}.shape'
                if not os.path.exists(fp):
                    continue
                n_unit += 1
                try:
                    with bz2.BZ2File(fp) as f:
                        outs = pickle.load(f)
                    gt = _SOL[tid][ti_idx] if tid in _SOL else None
                    if gt is not None and any(
                            restore_any(o['solution'], sp,
                                        te['input']).tolist() == gt
                            for o in outs):
                        n_hit += 1
                        print(f'  [적중] {key}_{ti_idx} ({tag})')
                except Exception:
                    pass
        print(f'오프라인: 복원 후 GT 일치 후보 보유 유닛 {n_hit}/{n_unit}')

예산 모드: 커밋 테스트(1h) - 항등 2100s / U형 700s
U형 쌍둥이 39개 (26태스크): {'Umono': 21, 'U3c': 3, 'U2c': 2, 'Umono_a': 13} / 항등 120태스크


2026-07-30 22:43:45.844146: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785451426.247753     171 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785451426.353400     171 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.00s/it]
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


항등 페이즈: 어댑터 병합 없음 (순수 sft139)
TTT-lite 준비: LoRA r=32, 30스텝/쌍둥이
  TTT 20270e3bo0zzz 37s
  TTT 28a6681fo0zzz 44s
  TTT d35bdbdco0zzz 45s
  TTT 78332cb0o0zzz 45s
  TTT e8686506o0zzz 30s
  TTT dbff022co0zzz 47s
  TTT eee78d87o0zzz 64s
  TTT 4c3d4a41o0zzz 51s
  TTT b0039139o0zzz 46s
  TTT dd6b8c4bo0zzz 53s
  TTT f931b4a8o0zzz 70s
  TTT 7b5033c1o0zzz 43s
  TTT 3dc255dbo0zzz 57s
  TTT 16de56c4o0zzz 60s
  TTT 97d7923eo0zzz 64s
  TTT bf45cf4bo0zzz 67s
시간 예산 소진 - 136b0064o0zzz 부터 생략
추론 완료: 24유닛 / 후보 380개 덤프 (2144s, TTT=True)
[항등] 페이즈 종료 rc=0 (2367s)


2026-07-30 23:22:16.019739: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785453736.043335     430 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785453736.050917     430 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 19.47it/s]
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


LoRA 병합 253층 <- shapeuni_shu2val.pt
TTT-lite 준비: LoRA r=32, 30스텝/쌍둥이
  TTT 28a6681fu0zzz 45s
  TTT 28a6681fu1zzz 45s
  TTT 28a6681fu2zzz 45s
  TTT 31f7f899u0zzz 66s
  TTT 2b83f449u0zzz 65s
  TTT 5545f144u0zzz 83s
시간 예산 소진 - 7666fa5du0zzz 부터 생략
추론 완료: 6유닛 / 후보 96개 덤프 (788s, TTT=True)
[U형] 페이즈 종료 rc=0 (869s)
  [적중] d35bdbdco0zzz_1 (orig)
  [적중] d35bdbdco0zzz_2 (orig)
  [적중] 78332cb0o0zzz_0 (orig)
  [적중] e8686506o0zzz_0 (orig)
  [적중] 4c3d4a41o0zzz_0 (orig)
  [적중] 4c3d4a41o0zzz_1 (orig)
  [적중] 3dc255dbo0zzz_0 (orig)
  [적중] 28a6681fu1zzz_0 (U3c/단색)
  [적중] 31f7f899u0zzz_0 (Umono_a)
오프라인: 복원 후 GT 일치 후보 보유 유닛 9/30


In [2]:
# ============================================================
# [우리단독 제출 셀 v2] 후보(항등+U형)로 submission.json 생성 (셀2, 마지막).
#   v1.1 -> v2 (어댑터 셀 v3 와 짝):
#     ① 항등('I') 후보 지원 - 복원 없이 그대로 (순수 sft139 원본 예측).
#     ② 복원 후 프록시색 잔존 후보 패널티(+1) - 색칠 실패분 강등.
#     (score_aug 는 셀1 v3 부터 토큰당 NLL - 득표와 스케일 정합)
#   짝: 우리단독_모양어댑터_셀.py (셀1). 1등 solver 불필요.
#   attempt_1 = 1등 / attempt_2 = 2등 (없으면 1등 복사).
#   v1.1: 순위 = 득표 - 평균 score_aug(NLL) (kgmon 식. 어댑터 셀 v2 가
#   후보별 증강 NLL 을 실측해 줌. 구덤프의 1.0 상수와도 호환).
#   후보 없는 유닛은 [[0]]. 자기검증(S/U 가설 자기모순 후보 폐기) 포함.
#   오프라인이면 정답 대조 점수 출력.
# ============================================================
import os, json, bz2, pickle
import numpy as np


def _G(x):
    return np.asarray(x, dtype=int)

def _shapes(g, bg, conn=4):
    g = _G(g); H, W = g.shape
    NB = ((1, 0), (-1, 0), (0, 1), (0, -1)) if conn == 4 else \
         tuple((a, b) for a in (-1, 0, 1) for b in (-1, 0, 1) if (a, b) != (0, 0))
    seen = np.zeros((H, W), bool); out = []
    for y in range(H):
        for x in range(W):
            if seen[y, x] or g[y, x] == bg:
                continue
            c0 = int(g[y, x]); st = [(y, x)]; seen[y, x] = True; cs = []
            while st:
                a, b = st.pop(); cs.append((a, b))
                for dy, dx in NB:
                    ny, nx = a + dy, b + dx
                    if 0 <= ny < H and 0 <= nx < W and not seen[ny, nx] \
                            and g[ny, nx] == c0:
                        seen[ny, nx] = True; st.append((ny, nx))
            out.append((c0, cs))
    return out

def _norm(cs):
    y0 = min(y for y, _ in cs); x0 = min(x for _, x in cs)
    return frozenset((y - y0, x - x0) for y, x in cs)

def _center(cs):
    return (sum(y for y, _ in cs) / len(cs), sum(x for _, x in cs) / len(cs))

def _S_silkey(cs):
    return json.dumps(sorted(map(list, _norm(cs))))

def _S_sil_paint(p, i, bg, m, colors):
    if not colors:
        return
    pool = {}
    for c, cs in _shapes(i, bg):
        if c in colors:
            pool.setdefault(_norm(cs), []).append((c, cs))
    for cs in [cs for c, cs in _shapes(p, bg) if c == m]:
        cand = pool.get(_norm(cs), [])
        if not cand:
            continue
        if len({c for c, _ in cand}) == 1:
            used = cand[0]
        else:
            cy, cx = _center(cs)
            used = min(cand, key=lambda t: (_center(t[1])[0] - cy) ** 2
                                           + (_center(t[1])[1] - cx) ** 2)
        cand.remove(used)
        for y, x in cs:
            p[y, x] = used[0]

def _S_groups(spec):
    v = spec['변형']; F, M, V = spec['F'], spec['M'], spec['V']
    if v == '2색':
        return [], F + M + V
    if v == '3색':
        return F, M + V
    if v == '3색도형':
        return F + M, V
    return [], M + V

def _apply_S_grid(g, spec):
    g = _G(g).copy()
    G1, G2 = _S_groups(spec)
    if G1:
        g[np.isin(g, G1)] = spec['m1']
    if G2:
        g[np.isin(g, G2)] = spec['m2']
    return g

def _restore_S(pred, spec, orig_in):
    p = _G(pred).copy(); i = _G(orig_in); bg = spec['bg']
    G1, G2 = _S_groups(spec)
    m1, m2 = spec.get('m1'), spec['m2']
    same = (p.shape == i.shape)
    if m1 is not None and same:
        mask = (p == m1) & np.isin(i, spec['F'])
        p[mask] = i[mask]
    if m1 is not None:
        _S_sil_paint(p, i, bg, m1, [c for c in spec['M'] if c in G1])
    mode = spec['변동복원']
    if mode != '모양색':
        _S_sil_paint(p, i, bg, m2, G2)
    if mode == '모양색':
        f = spec['모양표']
        for cs in [cs for c, cs in _shapes(p, bg) if c == m2]:
            v = f.get(_S_silkey(cs))
            if v is not None:
                for y, x in cs:
                    p[y, x] = v
    if mode in ('위치', '위치단색') and same:
        mask = (p == m2) & np.isin(i, G2)
        p[mask] = i[mask]
    if mode == '전이' and same:
        f = spec['전이표']
        ys, xs = np.where(p == m2)
        for y, x in zip(ys.tolist(), xs.tolist()):
            v = f.get(str(int(i[y, x])))
            if v is not None:
                p[y, x] = v
    if mode in ('단색', '위치단색') and spec.get('단색') is not None:
        p[p == m2] = spec['단색']
    return p

def _restore_U(pred, spec, orig_in):
    p = _G(pred).copy(); i = _G(orig_in); m = spec['m']
    if p.shape == i.shape and spec['F']:
        mask = (p == m) & np.isin(i, spec['F'])
        p[mask] = i[mask]
    _S_sil_paint(p, i, 0, m,
                 sorted({int(c) for c in np.unique(i) if c != 0}))
    return p

def restore_any(pred, spec, orig_in):
    if spec['종류'] == 'I':
        return _G(pred)                        # 항등: 원본 예측 그대로
    if spec['종류'] == 'U':
        return _restore_U(pred, spec, orig_in)
    return _restore_S(pred, spec, orig_in)


# ---------------- 실행: 우리 후보 -> submission.json ----------------
_rerun = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
_BASE = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/"
_SRC = _BASE + ("arc-agi_test_challenges.json" if _rerun
                else "arc-agi_evaluation_challenges.json")
_CH = json.load(open(_SRC))
try:
    _plan = json.load(open("/kaggle/working/분해_계획.json"))
except Exception:
    _plan = {}

def _load_store(path):
    d = {}
    for _key in (os.listdir(path) if os.path.isdir(path) else []):
        try:
            with bz2.BZ2File(os.path.join(path, _key)) as _f:
                _outs = pickle.load(_f)
        except Exception:
            continue
        _bk = _key.split(".")[0]
        d.setdefault(_bk, {})
        for _i, _sm in enumerate(_outs):
            d[_bk][f"{_key}.out{_i}"] = _sm
    return d
_virt = _load_store("/kaggle/inference_2step")

_sub = {tid: [{"attempt_1": [[0]], "attempt_2": [[0]]}
              for _ in _CH[tid]['test']] for tid in _CH}
_n_fill = _n_cand = 0
for _tid, _twins in _plan.items():
    if _tid not in _CH:
        continue
    for _k in range(len(_CH[_tid]['test'])):
        _ti = _CH[_tid]['test'][_k]['input']
        _pool = {}                            # 복원판 -> [표, 그리드]
        for _tw in _twins:
            _v = _virt.get(f"{_tw['k']}_{_k}")
            if not _v:
                continue
            _sp = _tw['spec']
            for _sm in _v.values():
                try:
                    _pg = np.asarray(_sm['solution'])
                    _tia = np.asarray(_ti)
                    # 자기검증: 가설이 예측에서 깨지면 폐기
                    if _sp['종류'] == 'U':
                        if _sp['F'] and _pg.shape == _tia.shape:
                            _iF = np.isin(_tia, _sp['F'])
                            if not (_pg[_iF] == _sp['m']).all():
                                continue
                    elif _sp['종류'] == 'S':
                        if _sp.get('m1') is not None and \
                           _pg.shape == _tia.shape:
                            _iF = np.isin(_tia, _sp['F'])
                            _cm1 = int((_apply_S_grid(_tia, _sp)
                                        == _sp['m1']).sum())
                            if not (_pg[_iF] == _sp['m1']).all() or \
                               int((_pg == _sp['m1']).sum()) != _cm1:
                                continue
                    _rg = restore_any(_pg, _sp, _ti).tolist()
                    _pen = 0.0                 # 복원 실패(프록시색 잔존) 강등
                    _ra = np.asarray(_rg)
                    if _sp['종류'] == 'U' and int((_ra == _sp['m']).sum()):
                        _pen = 1.0
                    elif _sp['종류'] == 'S' and \
                            int((_ra == _sp['m2']).sum()):
                        _pen = 1.0
                except Exception:
                    continue
                _h = json.dumps(_rg)
                _e = _pool.setdefault(_h, [0, 0.0, _rg])
                _e[0] += 1
                _e[1] += float(np.mean(_sm.get('score_aug', [1.0]))) + _pen
                _n_cand += 1
        if not _pool:
            continue
        _ranked = sorted(_pool.values(),
                         key=lambda x: -(x[0] - x[1] / x[0]))
        _slot = _sub[_tid][_k]
        _slot["attempt_1"] = _ranked[0][2]
        _slot["attempt_2"] = _ranked[1][2] if len(_ranked) > 1 \
                             else _ranked[0][2]
        _n_fill += 1

for _tid in _CH:                              # 구조 검증
    assert _tid in _sub and len(_sub[_tid]) == len(_CH[_tid]['test'])
    for _slot in _sub[_tid]:
        for _a in ("attempt_1", "attempt_2"):
            assert isinstance(_slot[_a], list) and _slot[_a] \
                   and isinstance(_slot[_a][0], list)
with open("submission.json", "w") as _f:
    json.dump(_sub, _f)
_tot = sum(len(v) for v in _sub.values())
print(f"우리단독 제출: {_n_fill}/{_tot}유닛 채움 (후보 {_n_cand}개, "
      f"계획 {sum(len(v) for v in _plan.values())}쌍둥이/{len(_plan)}태스크)")

if not _rerun:                                # 오프라인 채점
    try:
        _SOL = json.load(open(_BASE + "arc-agi_evaluation_solutions.json"))
        _s = 0.0
        _hits = []
        for _tid, _v in _SOL.items():
            for _i, _r in enumerate(_v):
                if any(_sub[_tid][_i][_a] == _r
                       for _a in ("attempt_1", "attempt_2")):
                    _s += 1 / len(_v)
                    _hits.append(f"{_tid}_{_i}")
        print(f"오프라인 점수(우리단독): {_s:.2f} / 정답 유닛: {_hits}")
    except Exception as _e:
        print("오프라인 채점 생략:", _e)

우리단독 제출: 27/172유닛 채움 (후보 460개, 계획 159쌍둥이/120태스크)
오프라인 점수(우리단독): 3.67 / 정답 유닛: ['3dc255db_0', '4c3d4a41_0', '4c3d4a41_1', 'd35bdbdc_1', 'd35bdbdc_2', 'e8686506_0']
